In [4]:
from pageindex import PageIndexClient
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import os

In [2]:
load_dotenv()  # Load environment variables from .env file

True

In [5]:
pageindex_client = PageIndexClient(api_key=os.getenv("PAGEINDEX_API_KEY"))
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [10]:
PDF_URL = "https://arxiv.org/pdf/2603.15031"
data_dir = "./data"
PDF_PATH = Path(data_dir) / "2603.15031.pdf"

In [11]:
print("Uploading the PDF document to PageIndex...")
result = pageindex_client.submit_document(PDF_PATH)

doc_id = result["doc_id"]
print(f"Document uploaded with doc_id: {doc_id}")

Uploading the PDF document to PageIndex...
Document uploaded with doc_id: pi-cmnla57zg0h9p01qpc44imqnp


In [12]:
print("Building tree index... ")

while True:
    status_result = pageindex_client.get_document(doc_id)
    status = status_result["status"]
    print(f"Current document status: {status}")

    if status == "completed":
        print("Document processing completed.")
        break
    elif status == "failed":
        print("Document processing failed.")
        exit(1)
    else:
        print("Document is still processing. Waiting for 10 seconds before checking again...")
        time.sleep(10)

Building tree index... 
Current document status: completed
Document processing completed.


In [14]:
import json
tree_result = pageindex_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result",[])

print(f"Top-level sections: {len(pageindex_tree)}")
print("Raw tree structure (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top-level sections: 14
Raw tree structure (first node):
{
  "title": "TECHNICAL REPORT OF ATTENTION RESIDUALS",
  "node_id": "0000",
  "page_index": 1,
  "summary": "# TECHNICAL REPORT OF ATTENTION RESIDUALS\n",
  "text": "# TECHNICAL REPORT OF ATTENTION RESIDUALS\n"
}


In [17]:
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = "  " * indent + ("--" if indent > 0 else "")
        page = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

In [18]:
print_tree(pageindex_tree)

[0000] TECHNICAL REPORT OF ATTENTION RESIDUALS (p.1)
[0001] Kimi Team (p.1)
  --[0002] ABSTRACT (p.1)
[0003] Notation. (p.3)
[0004] 2.1 Training Deep Networks via Residuals (p.3)
[0005] Residual Learning. (p.3)
[0006] Generalizing Residuals. (p.3)
[0007] Limitations. (p.3)
[0008] 3 Attention Residuals: A Unified View of Time and Depth (p.3)
  --[0009] The Duality of Time and Depth. (p.3)
  --[0010] Overhead. (p.4)
  --[0011] 3.2 Block Attention Residuals (p.4)
  --[0012] Intra-Block Accumulation. (p.4)
  --[0013] block_size counts #TTN + MLP; each transformer layer has 2 (p.5)
[0014] 4 Infrastructure Design (p.5)
  --[0015] 4.1 Training (p.6)
  --[0016] 4.2 Inference (p.7)
[0017] 5 Experiments (p.8)
  --[0018] 5.1 Scaling Laws (p.8)
  --[0019] 5.2 Main Results (p.9)
  --[0020] 5.3 Ablation Study (p.11)
  --[0021] 5.4 Analysis (p.12)
    --[0022] 5.4.1 Optimal Architecture (p.12)
    --[0023] 5.4.2 Analyzing Learned AttnRes Patterns (p.13)
[0024] 6 Discussions (p.14)
  --[0025] 6.1 Sequ

In [19]:
def count_nodes(nodes):
    count = 0
    for node in nodes:
        count += 1
        if node.get("nodes"):
            count += count_nodes(node["nodes"])
    return count

In [20]:
total_nodes = count_nodes(pageindex_tree)
print(f"Total nodes in the tree: {total_nodes}")

Total nodes in the tree: 31


In [25]:
def llm_tree_search(query:str, tree: list, model: str = "gpt-4o") -> dict:
    """
    Core PageIndex Retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the strucure and returns relevant node_ids.

    Returns: dict with "thinking" (reasoning) and "node_list" (node_ids)
    """

    def compress(nodes):
        compressed = []
        for node in nodes:
            entry = {
                "node_id": node["node_id"],
                "title": node["title"],
                "page_index": node.get("page_index", "?"),
                "summary": node.get("text", "")[:200]  # Include a short summary for context
            }
            if node.get("nodes"):
                entry["children"] = compress(node["nodes"])
            compressed.append(entry)
        return compressed

    compressed_tree = compress(tree)
    prompt = f"""You are given a query and document's tree structure (like a table of contents).
Your task is to identify which node IDs most likely contain the answer to the query.
Think step by step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<Your step-by-step reasoning here>",
  "node_list": ["node_id_1", "node_id_2", ...]  // List of node_ids that are relevant
}}
    """

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role":"user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

In [26]:
query = "Explain Attention Residuals in simple language."

print(f"Running LLM Tree Search for query: '{query}'")
result = llm_tree_search(query, pageindex_tree)
print("LLM Tree Search Result:")
print(result.get("thinking", "N/A"))
print()
print(f"Relevant node IDs: {result.get('node_list', [])}")

Running LLM Tree Search for query: 'Explain Attention Residuals in simple language.'
LLM Tree Search Result:
To find an explanation of Attention Residuals in simple language, we need to look for sections in the document that discuss Attention Residuals and potentially explain them in a descriptive manner. We can start by considering the titles and summaries of each section. The query asks for a simple explanation, so sections that provide an overview, introduction, or summary of Attention Residuals are more likely to be relevant than those delving into technical details or experiments.

1. Starting from the main title (node_id 0000), it does not provide specific information but acts as an overview of the document topic: Attention Residuals.

2. The section titled '3 Attention Residuals: A Unified View of Time and Depth' (node_id 0008) is crucial because it focuses directly on Attention Residuals and provides a layered view, suggesting an in-depth discussion.

3. Within this section, th

In [27]:
def find_nodes_by_ids(tree: list, target_ids:list) -> list:
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [28]:
def generate_answer(query: str, nodes: list, model: str = "gpt-4o") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer to the query.
    Instructs the LLM to cite section titles and page numbers.
    """

    if not nodes:
        return "No relevant sections found to answer the query."

    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}] \n"
            f"{node.get('text', 'Content not available')}"
        )
    context = "\n\n--\n\n".join(context_parts)

    prompt = f"""Your are an expert document analyst. Answer the question using Only the provided context.
For every claim you make, cite the section title and page number in parantheses. If the answer is not in the context, say you don't know.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:
"""
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role":"user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

In [29]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG Pipeline:
    
    Step 1: LLM Tree search to find relevant node_ids
    Step 2: Find nodes in the tree matching those node_ids
    Step 3: Generate an answer grounded in the retrieved nodes, with citations
    """

    if verbose:
        print("*" * 50)
        print(f"Query: {query}")
        print("*" * 50)
        

    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list", [])

    if verbose:
        print("LLM Tree Search Reasoning:")
        print(f"{search_result.get('thinking', '')[:200]}...")
        print(f"Relevant node IDs: {node_ids}")
        print("-" * 50)

    relevant_nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"Retrieved {len(relevant_nodes)} relevant nodes for answer generation.")
        for node in relevant_nodes:
            print(f"- {node['title']} (p.{node.get('page_index', '?')})")
    answer = generate_answer(query, relevant_nodes)

    if verbose:
        print("-" * 50)
        print("Generated Answer:")
        print(answer)
    return answer

In [30]:
answer = vectorless_rag(query, pageindex_tree)

**************************************************
Query: Explain Attention Residuals in simple language.
**************************************************
LLM Tree Search Reasoning:
To find sections that likely explain 'Attention Residuals' in simple terms, I should start by looking for sections that introduce or provide an overview of Attention Residuals. In the table of content...
Relevant node IDs: ['0008', '0009']
--------------------------------------------------
Retrieved 2 relevant nodes for answer generation.
- 3 Attention Residuals: A Unified View of Time and Depth (p.3)
- The Duality of Time and Depth. (p.3)
--------------------------------------------------
Generated Answer:
Attention Residuals are a method for managing information in deep neural networks by applying attention mechanisms over the depth of the network, rather than over time as in sequence models. The approach allows each layer to selectively access outputs from all previous layers using attention weights, e

In [ ]:
response = pageindex_client.chat_completions(
    messages = [{"role":"user", "content": query}],
    doc_id = doc_id
)

AttributeError: 'dict' object has no attribute 'choices'

In [34]:
answer = response['choices'][0]['message']['content'].strip()
print("Answer from PageIndex API:")
print(answer)

Answer from PageIndex API:
I'll get the document structure first to locate the key sections explaining Attention Residuals.## Attention Residuals in Simple Language

**The Problem with Standard Residuals:**

In deep neural networks, residual connections add together outputs from all previous layers with equal weight (like 1+1+1+1...). This creates issues:
- Each layer's contribution gets diluted as the network gets deeper
- Later layers can't selectively retrieve useful information from earlier layers
- Hidden states grow uncontrollably, destabilizing training

**The Attention Residuals Solution:**

Instead of blindly adding all layer outputs equally, **Attention Residuals lets each layer *choose* which previous layers to pay attention to**. It's like the difference between:
- **Standard**: "Mix everything together equally"
- **AttnRes**: "Look at all previous layers and decide which ones are most relevant right now"

**The Key Analogy:**

Think of how transformers handle sequences of 

In [46]:
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

current = None

stream = await async_client.responses.create(
    model="gpt-5.4",
    stream = True,
    reasoning = {"effort": "medium", "summary": "auto"},
    instructions = "You are a document analysis assistant. Always use the available tools to search and retrieve information from the user's documents. Start by finding relevant documents, then get their structure and content.",
    tools = [
        {
            "type": "mcp",
            "server_label": "PageIndex",
            "server_url": "https://api.pageindex.ai/mcp",
            "headers": {"Authorization": f"Bearer {os.getenv('PAGEINDEX_API_KEY')}"},
            "require_approval": "never",
        }
    ],
    input = query
)

async for event in stream:
    if event.type == "response.output_item.added":
        if event.item.type == "reasoning":
            current = "reasoning"
            print("[reasoning] ", end="", flush=True)
        elif event.item.type == "mcp_list_tools":
            print(f"Discovered MCP tools\n")
    elif event.type == "response.reasoning_summary_part.done":
        print("...\n")
    elif event.type == "response.reasoning_summary_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.output_item.done":
        if event.item.type == "mcp_call":
            current = "tool"
            print(f"[tool_use] {event.item.name} ({event.item.arguments})\n")
            print(f"[tool result] {str(event.item.output)[:200]}...\n")
    elif event.type == "response.output_text.delta":
        if current != "answer":
            current = "answer"
            print("[answer]")
        print(event.delta, end="", flush=True)

Discovered MCP tools

[reasoning] [answer]
“Attention residuals” usually refers to the **skip connection** around an attention layer in a transformer.

Simple way to think about it:

- The model has some current understanding of the text.
- The **attention layer** adds new information by looking at other words.
- The **residual connection** says:  
  **don’t throw away the old understanding — keep it, and add the new attention output on top.**

In equation form, it’s basically:

**new state = old state + attention output**

### Why do this?
Because it helps the model:

- **keep useful original information**
- **learn small improvements instead of rebuilding everything**
- **train more easily and more stably**

### Analogy
Imagine you’re editing a draft:

- **Old draft** = current representation
- **Editor’s suggestions** = attention output
- **Residual connection** = keep the draft and apply the suggestions, instead of rewriting from scratch

So attention residuals are just the model’s